# Importing Libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, accuracy_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping

2026-04-04 22:36:22.781159: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775342183.012849      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775342183.079682      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775342183.620577      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775342183.620616      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775342183.620619      24 computation_placer.cc:177] computation placer alr

# Defining Functions

In [2]:
def data_generator(X, y, batch_size=32):
    """
    Yields dense batches of data from a sparse matrix to save memory.
    """
    num_samples = X.shape[0]
    
    # Convert pandas Series to numpy arrays for easier slicing
    if isinstance(y, pd.Series):
        y = y.values
        
    while True:
        # Shuffle indices at the start of each epoch for better training
        indices = np.random.permutation(num_samples)
        
        for start in range(0, num_samples, batch_size):
            end = min(start + batch_size, num_samples)
            batch_idx = indices[start:end]
            
            # Slice the sparse matrix, then convert ONLY this tiny batch to dense
            X_batch = X[batch_idx].toarray()
            y_batch = y[batch_idx]
            
            yield X_batch, y_batch

# Readind data

In [3]:
print("Loading data...")
# Replace these filenames with your actual file paths
train_df  = pd.read_parquet('/kaggle/input/datasets/ayusheinsteinyadav/original/train.parquet')
val_df    = pd.read_parquet('/kaggle/input/datasets/ayusheinsteinyadav/original/val.parquet')
test_df   = pd.read_parquet('/kaggle/input/datasets/ayusheinsteinyadav/original/test.parquet')
print("Data Loaded")

Loading data...
Data Loaded


In [4]:
num_classes = train_df['generated_by'].unique().tolist().__len__()
num_classes

12

# Splitting Data

In [5]:
text_col = 'text'
label_col = 'generated_by'

In [6]:
X_train_raw,  y_train  = train_df[text_col],  train_df[label_col]
X_val_raw,    y_val    = val_df[text_col],    val_df[label_col]
X_test_raw,   y_test   = test_df[text_col],   test_df[label_col]

# Encoding Labels

In [7]:
from sklearn.preprocessing import LabelEncoder

print("Encoding labels to integers...")
# Initialize the encoder
label_encoder = LabelEncoder()

# Fit on the training labels and transform them to integers
y_train = label_encoder.fit_transform(y_train)

# Transform validation and test labels based on what it learned from training
y_val = label_encoder.transform(y_val)
y_test = label_encoder.transform(y_test)

# We can also dynamically figure out how many classes you have!
num_classes = len(label_encoder.classes_)
print(f"Detected {num_classes} unique classes: {label_encoder.classes_}")
print('Labels Encoded Successfully')

Encoding labels to integers...
Detected 12 unique classes: ['chatgpt' 'cohere' 'cohere-chat' 'gpt2' 'gpt3' 'gpt4' 'human'
 'llama-chat' 'mistral' 'mistral-chat' 'mpt' 'mpt-chat']
Labels Encoded Successfully


# TF-IDF Tokenization

In [8]:
print("Applying TF-IDF...")
# Limiting max_features to save memory when converting to dense arrays later
tfidf = TfidfVectorizer(max_features=5000) 
    
# Fit ONLY on training data to avoid data leakage, then transform all
X_train_tfidf  = tfidf.fit_transform(X_train_raw)
X_val_tfidf    = tfidf.transform(X_val_raw)
X_test_tfidf   = tfidf.transform(X_test_raw)
print("TF-IDF Applied")

Applying TF-IDF...
TF-IDF Applied


# Embeding Model Placeholder

In [9]:
X_train_final  = X_train_tfidf
X_val_final    = X_val_tfidf
X_test_final   = X_test_tfidf

# Neural Network Hyperparameter Tuning

In [10]:
print("Training and tuning Neural Networks...")
    
# List of tuples defining the sizes of (Hidden Layer 1, Hidden Layer 2)
layer_configs = [(32, 16), (64, 32), (128, 64), (256, 128), (512, 256), (1024, 512), (2048, 1024), (4096, 2048), (8192, 4096)]

# CHANGE 1: We now want the LOWEST loss, so we initialize with infinity
best_val_loss = float('inf') 
best_val_acc = 0.0 # We keep this just so we can report the accuracy of the best model
best_config = None
best_model = None

# Determine input shape dynamically based on your final features
input_dim = X_train_final.shape[1]

for config in layer_configs:
    neurons_l1, neurons_l2 = config
    print(f"\nTraining config: Layer 1 = {neurons_l1} neurons, Layer 2 = {neurons_l2} neurons")
        
    # Build the 2-hidden-layer model
    model = Sequential([
        Input(shape=(input_dim,)), 
        
        Dense(neurons_l1, activation='relu'),
        BatchNormalization(), # <-- Helps the model train faster and more stably
        Dropout(0.3),         # <-- Slightly higher dropout to prevent overfitting
        
        Dense(neurons_l2, activation='relu'),
        BatchNormalization(), 
        Dropout(0.3),
        
        Dense(num_classes, activation='softmax') 
    ])

    # Compile the model
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
        
    batch_size = 32
        
    # We have to tell Keras how many batches make up one full epoch
    steps_per_epoch = int(np.ceil(X_train_final.shape[0] / batch_size))
    val_steps = int(np.ceil(X_val_final.shape[0] / batch_size))

    # CHANGE 2: Create the EarlyStopping callback
    early_stop = EarlyStopping(
        monitor='val_loss',       # Watch the validation loss
        patience=2,               # Optional: Stop early if it doesn't improve for 2 epochs
        restore_best_weights=True # Rewind to the exact epoch with the lowest val_loss!
    )

    # Train the model using the generator AND the callback
    # Train the model using the generator AND the callback
    model.fit(
        data_generator(X_train_final, y_train, batch_size),
        validation_data=data_generator(X_val_final, y_val, batch_size),
        steps_per_epoch=steps_per_epoch,
        validation_steps=val_steps,
        epochs=50, # <-- INCREASE THIS!
        callbacks=[early_stop], 
        verbose=0
    )
        
    # Evaluate on validation set using the generator
    # Because of restore_best_weights, this evaluates the BEST version of this model, not just the last epoch
    val_loss, val_acc = model.evaluate(
        data_generator(X_val_final, y_val, batch_size), 
        steps=val_steps, 
        verbose=0 # Set to 0 to keep the console slightly cleaner
    )
    print(f"Validation Loss: {val_loss:.4f} | Validation Accuracy: {val_acc:.4f}")

    # CHANGE 4: Keep track of the best model based on MINIMUM loss
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_val_acc = val_acc # Saving this to print it at the end
        best_config = config
        best_model = model

print(f"\n==========================================")
print(f"BEST CONFIGURATION FOUND (Based on Lowest Val Loss):")
print(f"Layer 1: {best_config[0]} neurons | Layer 2: {best_config[1]} neurons")
print(f"Best Validation Loss: {best_val_loss:.4f}")
print(f"Associated Validation Accuracy: {best_val_acc:.4f}")
print(f"==========================================\n")

Training and tuning Neural Networks...

Training config: Layer 1 = 32 neurons, Layer 2 = 16 neurons


I0000 00:00:1775342368.384977      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1775342368.390984      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1775342371.370563      77 service.cc:152] XLA service 0x7ca8c4003060 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1775342371.370602      77 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1775342371.370608      77 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1775342371.737459      77 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1775342373.808221      77 device_compiler.h:188] Compiled clust

Validation Loss: 1.2266 | Validation Accuracy: 0.5770

Training config: Layer 1 = 64 neurons, Layer 2 = 32 neurons
Validation Loss: 1.0603 | Validation Accuracy: 0.6322

Training config: Layer 1 = 128 neurons, Layer 2 = 64 neurons
Validation Loss: 0.9127 | Validation Accuracy: 0.6840

Training config: Layer 1 = 256 neurons, Layer 2 = 128 neurons
Validation Loss: 0.8279 | Validation Accuracy: 0.7209

Training config: Layer 1 = 512 neurons, Layer 2 = 256 neurons
Validation Loss: 0.7794 | Validation Accuracy: 0.7406

Training config: Layer 1 = 1024 neurons, Layer 2 = 512 neurons
Validation Loss: 0.7517 | Validation Accuracy: 0.7523

Training config: Layer 1 = 2048 neurons, Layer 2 = 1024 neurons
Validation Loss: 0.7297 | Validation Accuracy: 0.7631

Training config: Layer 1 = 4096 neurons, Layer 2 = 2048 neurons
Validation Loss: 0.7140 | Validation Accuracy: 0.7626

Training config: Layer 1 = 8192 neurons, Layer 2 = 4096 neurons
Validation Loss: 0.7275 | Validation Accuracy: 0.7648

BEST 

# Evaluate on Test Data

In [11]:
print("Evaluating the best model on unseen Test Data...")
    
# Get probability predictions
y_pred_probs = best_model.predict(X_test_final)
    
# Convert probabilities to 0 or 1 (threshold at 0.5)
# (Note: For multi-class, use y_pred = np.argmax(y_pred_probs, axis=1))
y_pred = np.argmax(y_pred_probs, axis=1)

# Print final metrics
test_acc = accuracy_score(y_test, y_pred)
print(f"Final Test Accuracy: {test_acc:.4f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred))

Evaluating the best model on unseen Test Data...
3113/3113 ━━━━━━━━━━━━━━━━━━━━ 249s 80ms/step
Final Test Accuracy: 0.7626

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.82      0.85      8256
           1       0.81      0.73      0.77      8425
           2       0.78      0.73      0.75      8339
           3       0.80      0.73      0.76      8294
           4       0.80      0.81      0.81      8185
           5       0.91      0.84      0.87      8273
           6       0.46      0.93      0.61      8242
           7       0.87      0.81      0.84      8430
           8       0.72      0.67      0.70      8212
           9       0.86      0.66      0.75      8319
          10       0.84      0.68      0.76      8290
          11       0.81      0.73      0.77      8335

    accuracy                           0.76     99600
   macro avg       0.79      0.76      0.77     99600
weighted avg       0.79      0.76      0.